# 循环神经网络 RNN：记住过去的信息
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：循环神经网络_RNN.py | 核心功能：序列建模、隐状态传递、梯度消失问题

## 概述

RNN 专门为序列数据设计。它在每个时间步接收当前输入和上一步的隐状态，输出新的隐状态。这让 RNN 有"记忆"——能记住序列中之前的信息。

核心公式：h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b)

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 数学原理

### 1. RNN 的递推公式

**代码对应**：RNN 在时间步 $t$ 的隐藏状态更新：

$$\mathbf{h}_t = \tanh(\mathbf{W}_{xh}\mathbf{x}_t + \mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{b}_h)$$

$$\hat{\mathbf{y}}_t = \mathbf{W}_{hy}\mathbf{h}_t + \mathbf{b}_y$$

其中 $\mathbf{W}_{xh}$ 为输入到隐藏的权重，$\mathbf{W}_{hh}$ 为隐藏到隐藏的权重（循环连接）。

### 2. 通过时间的反向传播（BPTT）

梯度沿时间步反向传播：

$$\frac{\partial L}{\partial \mathbf{W}_{hh}} = \sum_{t=1}^{T}\frac{\partial L_t}{\partial \mathbf{W}_{hh}} = \sum_{t=1}^{T}\sum_{k=1}^{t}\frac{\partial L_t}{\partial \mathbf{h}_t}\left(\prod_{j=k+1}^{t}\frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}\right)\frac{\partial \mathbf{h}_k}{\partial \mathbf{W}_{hh}}$$

其中 $\frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}} = \text{diag}(\tanh'(\mathbf{z}_j))\mathbf{W}_{hh}$。

### 3. 梯度消失/爆炸

$\prod_{j=k+1}^{t}\frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}} \approx \mathbf{W}_{hh}^{t-k}$（忽略激活函数导数）

- $\|\mathbf{W}_{hh}\| < 1$：梯度指数衰减（梯度消失）→ 长距离依赖无法学习
- $\|\mathbf{W}_{hh}\| > 1$：梯度指数增长（梯度爆炸）→ 训练不稳定

**梯度裁剪**（Gradient Clipping）缓解爆炸：$\mathbf{g} \leftarrow \min(1, \frac{c}{\|\mathbf{g}\|})\mathbf{g}$

这就是 LSTM 和 GRU 被发明的根本原因。

### 1. 序列数据生成

运行 1. 序列数据生成 的代码，观察算法在该环节的行为。

In [ ]:
# 任务：根据正弦波序列预测下一个值
np.random.seed(42)
timesteps = 100
n_samples = 500

def generate_sine_data(n_samples, timesteps):
    X, y = [], []
    for _ in range(n_samples):
        start = np.random.uniform(0, 2 * np.pi)
        t = np.linspace(start, start + 4 * np.pi, timesteps + 1)
# --- 数值计算 ---
        data = np.sin(t) + 0.1 * np.randn(len(t))  # 加噪声
        X.append(data[:-1])  # 输入: 前 100 个点
        y.append(data[-1])   # 目标: 第 101 个点
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = generate_sine_data(n_samples, timesteps)
# 划分训练/测试
X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]

# 转为 (batch, seq_len, features) 形状
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1)  # (400, 100, 1)
X_test_t = torch.FloatTensor(X_test).unsqueeze(-1)
y_train_t = torch.FloatTensor(y_train)
y_test_t = torch.FloatTensor(y_test)

### 2. 定义 RNN 模型

运行 2. 定义 RNN 模型 的代码，观察算法在该环节的行为。

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
# --- 继续 ---
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # h0: 初始隐状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)  # out: (batch, seq_len, hidden_size)
        out = self.fc(out[:, -1, :])  # 取最后一个时间步的输出
        return out.squeeze(-1)

model = RNNModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"=== RNN 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 50
for epoch in range(n_epochs):
    model.train()
    output = model(X_train_t)
# --- 继续 ---
    loss = criterion(output, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    # 梯度裁剪：防止梯度爆炸
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test_t)
            test_loss = criterion(test_pred, y_test_t)
# --- 继续 ---
            train_rmse = torch.sqrt(loss).item()
            test_rmse = torch.sqrt(test_loss).item()
        print(f"  Epoch {epoch+1:>2}: Train RMSE={train_rmse:.4f}, Test RMSE={test_rmse:.4f}")

### 4. 预测效果

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测效果（前 10 个测试样本）===")
model.eval()
with torch.no_grad():
    preds = model(X_test_t).numpy()
for i in range(10):
# --- 输出结果 ---
    print(f"  真实={y_test[i]:>8.4f}, 预测={preds[i]:>8.4f}, "
          f"误差={abs(y_test[i]-preds[i]):>8.4f}")

### 5. RNN 的梯度问题

运行 5. RNN 的梯度问题 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== RNN 的梯度问题 ===")
print("梯度消失: 长序列中，梯度在反向传播时指数衰减，导致远距离依赖难以学习")
print("梯度爆炸: 梯度在反向传播时指数增长，导致训练不稳定")

# 演示梯度消失
print("\n梯度范数示例:")
for name, param in model.named_parameters():
    if "weight" in name and param.grad is not None:
        print(f"  {name}: grad_norm={param.grad.norm().item():.6f}")

### 6. LSTM 和 GRU 解决方案

运行 6. LSTM 和 GRU 解决方案 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 解决方案: LSTM / GRU ===")
print("LSTM: 引入门控机制（遗忘门、输入门、输出门）和细胞状态，缓解梯度消失")
print("GRU: LSTM 的简化版（重置门、更新门），参数更少，效果相近")

print("\n=== RNN 要点 ===")
print("- 输入: (batch, seq_len, input_size)")
print("- 隐状态 h_t = tanh(W_hh * h_{t-1} + W_xh * x_t)")
print("- 适合：文本、时间序列、语音等变长序列数据")
print("- 缺点：难以捕捉长距离依赖（被 LSTM/GRU/Transformer 取代）")

## 关键代码解释

### PyTorch RNN

```python
rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
output, hn = rnn(x)  # output: 所有时间步输出, hn: 最后隐状态
```

### 梯度消失问题

RNN 在长序列上训练时，梯度会指数级衰减（消失）或爆炸。这导致 RNN 难以学习长距离依赖。

## 注意事项

1. **梯度消失/爆炸**：长序列上严重，需要 LSTM/GRU 或梯度裁剪
2. **不能并行**：时间步必须串行计算
3. **短序列效果好**：长序列考虑 LSTM/Transformer

## 总结与延伸

以上代码展示了 **循环神经网络 RNN** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **双向 RNN**：同时从左到右和从右到左读序列
- **深层 RNN**：多层 RNN 堆叠
- **Transformer 的崛起**：自注意力完全替代 RNN 的序列建模能力
